# 10 - Post hoc amendment 2A - sensitivity to the Random Forest hyperparameters

**Post hoc.** The hyperparameters of the Random Forest were fixed in the analysis plan and never tuned. This notebook
re-runs the frozen pipeline with a small grid of hyperparameters, one changed at a time, and reports every
configuration. No configuration is chosen: the point is to show whether the conclusions depend on the fixed
choice, not to find a better model (choosing one would need nested cross-validation and would be another study).

**Kept apart from the pre-specified analysis** exactly like amendment 1: the plan is not edited; the amendment has
its own file, config/post_hoc_plan_2.json, frozen in outputs/post_hoc/post_hoc_plan_2_frozen.json before
anything is computed; results go only to outputs/post_hoc/; no key number is recorded, so notebook 08 is unaffected.

**Check:** the frozen configuration goes through the same code as notebooks 02 and 05 and must reproduce their
predictions exactly; the notebook stops if it does not.

**Outputs:** outputs/post_hoc/hyperparameters.json, hyperparameter_table.csv (Table S11) and the note; the figure
(Fig. S5) is drawn by notebook 12.

**Time:** about the time of notebook 02 for each configuration (10 configurations; on the full data, roughly
20-40 minutes). Each configuration is cached: a second run takes seconds.

In [ ]:
# Setup: make the eegad package importable (also when it was not installed with `pip install -e .`)
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=RuntimeWarning)   # MNE notices about head size and 'boundary' events
warnings.filterwarnings("ignore", category=FutureWarning)

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "analysis_plan.json").exists()), None)
if _root is None:
    raise FileNotFoundError("Open this notebook from inside the repository (eeg-ad-demographics/notebooks).")
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

In [ ]:
import time

import numpy as np
import pandas as pd
from IPython.display import display

from eegad.config import load_context
from eegad.data import load_table
from eegad.hyperparams import (HYPERPARAMETER_NOTE, configurations, loso_config, sensitivity_table, transport_config,
                               transport_metrics, within_metrics)
from eegad.posthoc import POST_HOC_PLAN_2, freeze_post_hoc_plan
from eegad.reporting import write_json

ctx = load_context("10")
out = ctx.out
PH, PH_HASH, PH_FROZEN = freeze_post_hoc_plan(ctx, POST_HOC_PLAN_2)
FOLDER = out / "post_hoc"
df_bl, df_gr = load_table(out / "derived" / "brainlat_epochs.csv"), load_table(out / "derived" / "greek_epochs.csv")
subj_bl, subj_gr = load_table(out / "derived" / "brainlat_subjects.csv"), load_table(out / "derived" / "greek_subjects.csv")
CONFIGS = configurations(ctx, PH["A_hyperparameter_sensitivity"]["grid"])
NB, SEED = ctx.n_boot, ctx.seed
print(f"Amendment {PH['amendment']}: {len(CONFIGS)} configurations | bootstrap {NB} | mode {ctx.tag}")
display(pd.DataFrame([{"configuration": c["label"], **c["params"]} for c in CONFIGS]))

## The grid (the frozen configuration first, checked against notebooks 02 and 05)

In [ ]:
REF = {"BrainLat": load_table(out / "predictions" / "oof_BrainLat_RF_11.csv"),
       "Greek": load_table(out / "predictions" / "oof_Greek_RF_11.csv"),
       "RQ3a": load_table(out / "predictions" / "transport_RQ3a.csv"),
       "RQ3b": load_table(out / "predictions" / "transport_RQ3b.csv")}


def same(a, b):
    m = a[["subject_id", "p"]].merge(b[["subject_id", "p"]], on="subject_id", suffixes=("", "_ref"))
    return len(m) == len(b) and np.allclose(m["p"], m["p_ref"], rtol=0, atol=1e-12)


RESULTS = []
t_all = time.time()
for i, c in enumerate(CONFIGS):
    t0 = time.time()
    tag = f"HP{i:02d}"
    pred = {"BrainLat": loso_config(df_bl, ctx.features, c["params"], f"{tag}_BL_{ctx.tag}", ctx),
            "Greek": loso_config(df_gr, ctx.features, c["params"], f"{tag}_GR_{ctx.tag}", ctx),
            "RQ3a": transport_config(df_bl, df_gr, ctx.features, c["params"], f"{tag}_RQ3a_{ctx.tag}", ctx),
            "RQ3b": transport_config(df_gr, df_bl, ctx.features, c["params"], f"{tag}_RQ3b_{ctx.tag}", ctx)}
    if c["frozen"]:
        bad = [k for k in pred if not same(pred[k], REF[k])]
        if bad:
            raise RuntimeError(f"The frozen configuration does not reproduce notebooks 02/05 for {bad}: stop.")
        print("Frozen configuration: reproduces the predictions of notebooks 02 and 05 exactly.")
    r = {"label": c["label"], "frozen": c["frozen"], "params": c["params"],
         "BrainLat": within_metrics(pred["BrainLat"], subj_bl, ctx.cov_brainlat, "BrainLat", NB, SEED),
         "Greek": within_metrics(pred["Greek"], subj_gr, ctx.cov_greek, "Greek", NB, SEED),
         "RQ3a": transport_metrics(pred["RQ3a"], subj_gr, NB, SEED),
         "RQ3b": transport_metrics(pred["RQ3b"], subj_bl, NB, SEED)}
    RESULTS.append(r)
    print(f"{c['label']:22s} AUC BL {r['BrainLat']['auc'][0]:.3f} GR {r['Greek']['auc'][0]:.3f} | OR/SD BL "
          f"{r['BrainLat']['OR']:.2f} GR {r['Greek']['OR']:.2f} | transport {r['RQ3a']['auc'][0]:.3f} / "
          f"{r['RQ3b']['auc'][0]:.3f} | {time.time() - t0:.0f} s")
print(f"All configurations in {(time.time() - t_all) / 60:.1f} min")

## Table S11 (no configuration is selected)

In [ ]:
S11 = sensitivity_table(RESULTS)
display(S11)
S11.to_csv(FOLDER / "hyperparameter_table.csv", index=False, encoding="utf-8-sig")
(FOLDER / "hyperparameter_note.txt").write_text(HYPERPARAMETER_NOTE.format(n_boot=NB), encoding="utf-8")
write_json(FOLDER / "hyperparameters.json", {"amendment": PH["amendment"], "post_hoc_plan_hash": PH_HASH,
                                             "frozen_at": PH_FROZEN, "amends_plan_hash": ctx.plan_hash,
                                             "mode": ctx.tag, "n_boot": NB, "results": RESULTS})
auc_bl = [r["BrainLat"]["auc"][0] for r in RESULTS]
auc_gr = [r["Greek"]["auc"][0] for r in RESULTS]
print(f"Range across the grid: AUC BrainLat {min(auc_bl):.3f}-{max(auc_bl):.3f}; Greek {min(auc_gr):.3f}-{max(auc_gr):.3f}")